Estoy intentando entender el flujo de ejecucion de un agnete

 La Clase Base (El "Contrato")

Primero, definimos la estructura mínima que todas las herramientas deben seguir.

In [31]:
from typing import Any, Dict

class Tool:
    name: str
    description: str
    inputs: Dict[str, Dict[str, Any]]
    output_type: str

    def forward(self, *args, **kwargs):
        pass

. Tu Herramienta Personalizada (El "Mock")

Vamos a crear una herramienta que simula buscar el precio de una acción (stock).

In [32]:
import datetime

class StockPriceTool(Tool):
    name = "get_stock_price"
    description = "Obtiene el precio de una acción. Si no existe, sugiere opciones."
    inputs = {'symbol': {'type': 'string', 'description': 'Símbolo de la acción'}}
    output_type = "string"

    def _log_activity(self, message):
        """Función interna para escribir en un archivo de log"""
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open("agent_log.txt", "a") as f:
            f.write(f"[{timestamp}] {message}\n")

    def forward(self, symbol: str) -> str:
        mock_data = {"AAPL": 175.5, "TSLA": 240.2, "GOOGL": 140.1}
        s = symbol.upper()
        
        if s in mock_data:
            res = f"El precio de {s} es {mock_data[s]} USD."
            self._log_activity(f"SUCCESS: {s} consultado.")
            return res
        else:
            opciones = ", ".join(mock_data.keys())
            msg = f"Error: '{s}' no encontrado. Intenta con: {opciones}"
            self._log_activity(f"ERROR: Fallo al buscar {s}.")
            return msg

# Prueba de nuevo
nasdaq_tool = StockPriceTool()
print(nasdaq_tool.forward("aaples"))

Error: 'AAPLES' no encontrado. Intenta con: AAPL, TSLA, GOOGL


El Motor del Agente (Emulación de smolagents)

Este es el código que "lee" tu clase y la convierte en algo que un LLM entendería.

In [33]:
class MockAgent:
    def __init__(self, tools):
        self.tools = {tool.name: tool for tool in tools}

    def generate_system_prompt(self):
        """Emula cómo el agente se presenta ante el LLM"""
        prompt = "Eres un asistente con acceso a las siguientes herramientas:\n"
        for name, tool in self.tools.items():
            prompt += f"\n- {name}: {tool.description}"
            prompt += f"\n  Args: {tool.inputs}\n"
        return prompt

    def handle_llm_call(self, tool_name: str, argument: str):
        """Emula el momento en que el LLM decide usar una herramienta"""
        if tool_name in self.tools:
            tool = self.tools[tool_name]
            print(f"--- Ejecutando {tool_name} con '{argument}' ---")
            # En la vida real, smolagents parsea los argumentos automáticamente
            result = tool.forward(argument)
            return f"Observation: El precio es {result}"
        return "Error: Herramienta no encontrada"

# --- INSTANCIACIÓN ---
my_tool = StockPriceTool()
agent = MockAgent(tools=[my_tool])

Ejecución del Flujo Completo

Ahora mira cómo se conectan las piezas:

In [34]:
# PASO 1: Ver qué lee el LLM (Metadatos)
print("### 1. LO QUE EL LLM VE (SYSTEM PROMPT) ###")
print(agent.generate_system_prompt())

# PASO 2: Simular que el LLM genera una acción tras pensar
print("\n### 2. ACCIÓN DEL AGENTE ###")
# El LLM diría internamente: "Necesito el precio de Apple, usaré get_stock_price"
respuesta_herramienta = agent.handle_llm_call("get_stock_price", "AAPL")

# PASO 3: Ver el resultado final
print(f"\n### 3. RESULTADO ###\n{respuesta_herramienta}")

### 1. LO QUE EL LLM VE (SYSTEM PROMPT) ###
Eres un asistente con acceso a las siguientes herramientas:

- get_stock_price: Obtiene el precio de una acción. Si no existe, sugiere opciones.
  Args: {'symbol': {'type': 'string', 'description': 'Símbolo de la acción'}}


### 2. ACCIÓN DEL AGENTE ###
--- Ejecutando get_stock_price con 'AAPL' ---

### 3. RESULTADO ###
Observation: El precio es El precio de AAPL es 175.5 USD.
